In [18]:
# fix_rvm_onnx_for_openvino_gpu.py
import subprocess
import sys
from pathlib import Path

import numpy as np
import onnx
from onnx import checker, numpy_helper, shape_inference

import openvino as ov


SRC = Path("models/rvm_mobilenetv3_fp32.onnx")
FIXED = Path("models/rvm_mobilenetv3_fp32_fixed.onnx")
SIM = Path("models/rvm_mobilenetv3_fp32_fixed_sim.onnx")

RATIO = 1.0

SHAPES = {
    "src": [1, 3, 320, 320],
    "r1i": [1, 16, 160, 160],
    "r2i": [1, 20, 80, 80],
    "r3i": [1, 40, 40, 40],
    "r4i": [1, 64, 20, 20],
    "fgr": [1, 3, 320, 320],
    "pha": [1, 1, 320, 320],
    "r1o": [1, 16, 160, 160],
    "r2o": [1, 20, 80, 80],
    "r3o": [1, 40, 40, 40],
    "r4o": [1, 64, 20, 20],
}


def set_value_info_shape(value_info, shape):
    dims = value_info.type.tensor_type.shape.dim
    del dims[:]
    for v in shape:
        d = dims.add()
        d.dim_value = int(v)


def fix_onnx():
    model = onnx.load(SRC)

    # remove downsample_ratio from graph inputs
    inputs = [i for i in model.graph.input if i.name != "downsample_ratio"]
    del model.graph.input[:]
    model.graph.input.extend(inputs)

    # remove existing initializer named downsample_ratio
    inits = [i for i in model.graph.initializer if i.name != "downsample_ratio"]
    del model.graph.initializer[:]
    model.graph.initializer.extend(inits)

    # add constant downsample_ratio
    model.graph.initializer.append(
        numpy_helper.from_array(
            np.array([RATIO], dtype=np.float32),
            name="downsample_ratio",
        )
    )

    # fix graph input/output shapes
    for vi in list(model.graph.input) + list(model.graph.output):
        if vi.name in SHAPES:
            set_value_info_shape(vi, SHAPES[vi.name])

    model = shape_inference.infer_shapes(model)
    checker.check_model(model)
    onnx.save(model, FIXED)
    print(f"[OK] saved fixed ONNX: {FIXED}")


def run_onnxsim():
    try:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "onnxsim",
                str(FIXED),
                str(SIM),
            ],
            check=True,
        )
        print(f"[OK] saved simplified ONNX: {SIM}")
        return SIM
    except Exception as e:
        print(f"[WARN] onnxsim failed, will use fixed ONNX directly: {e}")
        return FIXED


def check_openvino(path: Path):
    core = ov.Core()
    model = core.read_model(str(path))

    # force reshape again inside OpenVINO
    model.reshape({
        "src": [1, 3, 320, 320],
        "r1i": [1, 16, 160, 160],
        "r2i": [1, 20, 80, 80],
        "r3i": [1, 40, 40, 40],
        "r4i": [1, 64, 20, 20],
    })

    print("\n[OpenVINO inputs]")
    for inp in model.inputs:
        print(inp.get_any_name(), inp.partial_shape)

    print("\n[OpenVINO outputs]")
    for out in model.outputs:
        print(out.get_any_name(), out.partial_shape)

    print("\n[Dynamic ops]")
    count = 0
    for op in model.get_ordered_ops():
        for out in op.outputs():
            if out.partial_shape.is_dynamic:
                print(op.get_type_name(), op.get_friendly_name(), out.partial_shape)
                count += 1
                break

    if count == 0:
        print("[OK] no dynamic ops detected")
    else:
        print(f"[WARN] dynamic op count: {count}")

    xml_path = path.with_suffix(".xml")
    ov.save_model(model, str(xml_path))
    print(f"\n[OK] saved OpenVINO IR: {xml_path}")

    print("\n[Compile CPU]")
    core.compile_model(model, "CPU")
    print("[OK] CPU compile success")

    print("\n[Compile GPU]")
    core.compile_model(model, "GPU")
    print("[OK] GPU compile success")

In [19]:
fix_onnx()
final_onnx = run_onnxsim()
check_openvino(final_onnx)

[OK] saved fixed ONNX: models\rvm_mobilenetv3_fp32_fixed.onnx
[OK] saved simplified ONNX: models\rvm_mobilenetv3_fp32_fixed_sim.onnx

[OpenVINO inputs]
src [1,3,320,320]
r1i [1,16,160,160]
r2i [1,20,80,80]
r3i [1,40,40,40]
r4i [1,64,20,20]

[OpenVINO outputs]
fgr [1,3,320,320]
pha [1,1,320,320]
r1o [1,16,160,160]
r2o [1,20,80,80]
r3o [1,40,40,40]
r4o [1,64,20,20]

[Dynamic ops]
[OK] no dynamic ops detected

[OK] saved OpenVINO IR: models\rvm_mobilenetv3_fp32_fixed_sim.xml

[Compile CPU]
[OK] CPU compile success

[Compile GPU]
[OK] GPU compile success


In [ ]:
# fix_mediapipe_onnx_for_openvino.py
import subprocess
import sys
from pathlib import Path

import onnx
from onnx import checker, shape_inference

import openvino as ov


SRC = Path("models/mediapipe.onnx")
FIXED = Path("models/mediapipe_fixed.onnx")
SIM = Path("models/mediapipe_fixed_sim.onnx")

SHAPES = {
    "input_1:0": [1, 144, 256, 3],
    "segment:0": [1, 144, 256, 2],
}


def set_value_info_shape(value_info, shape):
    dims = value_info.type.tensor_type.shape.dim
    del dims[:]
    for v in shape:
        d = dims.add()
        d.dim_value = int(v)


def fix_onnx():
    model = onnx.load(SRC)

    for vi in list(model.graph.input) + list(model.graph.output):
        if vi.name in SHAPES:
            set_value_info_shape(vi, SHAPES[vi.name])

    model = shape_inference.infer_shapes(model)
    checker.check_model(model)
    onnx.save(model, FIXED)
    print(f"[OK] saved fixed ONNX: {FIXED}")


def run_onnxsim():
    try:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "onnxsim",
                str(FIXED),
                str(SIM),
            ],
            check=True,
        )
        print(f"[OK] saved simplified ONNX: {SIM}")
        return SIM
    except Exception as e:
        print(f"[WARN] onnxsim failed, will use fixed ONNX directly: {e}")
        return FIXED


def check_openvino(path: Path):
    core = ov.Core()
    model = core.read_model(str(path))

    model.reshape({
        "input_1:0": [1, 144, 256, 3],
    })

    print("\n[OpenVINO inputs]")
    for inp in model.inputs:
        print(inp.get_any_name(), inp.partial_shape)

    print("\n[OpenVINO outputs]")
    for out in model.outputs:
        print(out.get_any_name(), out.partial_shape)

    print("\n[Dynamic ops]")
    count = 0
    for op in model.get_ordered_ops():
        for out in op.outputs():
            if out.partial_shape.is_dynamic:
                print(op.get_type_name(), op.get_friendly_name(), out.partial_shape)
                count += 1
                break

    if count == 0:
        print("[OK] no dynamic ops detected")
    else:
        print(f"[WARN] dynamic op count: {count}")

    xml_path = path.with_suffix(".xml")
    ov.save_model(model, str(xml_path))
    print(f"\n[OK] saved OpenVINO IR: {xml_path}")

    print("\n[Compile CPU]")
    core.compile_model(model, "CPU")
    print("[OK] CPU compile success")

    print("\n[Compile GPU]")
    core.compile_model(model, "GPU")
    print("[OK] GPU compile success")


fix_onnx()
final_onnx = run_onnxsim()
check_openvino(final_onnx)

In [ ]:
# fix_mediapipe_new_onnx_for_openvino.py
import subprocess
import sys
from pathlib import Path

import onnx
from onnx import checker, shape_inference

import openvino as ov


SRC = Path("models/mediapipe_new.onnx")
FIXED = Path("models/mediapipe_new_fixed.onnx")
SIM = Path("models/mediapipe_new_fixed_sim.onnx")

SHAPES = {
    "pixel_values": [1, 3, 256, 256],
    "alphas": [1, 1, 256, 256],
}


def set_value_info_shape(value_info, shape):
    dims = value_info.type.tensor_type.shape.dim
    del dims[:]
    for v in shape:
        d = dims.add()
        d.dim_value = int(v)


def fix_onnx():
    model = onnx.load(SRC)

    for vi in list(model.graph.input) + list(model.graph.output):
        if vi.name in SHAPES:
            set_value_info_shape(vi, SHAPES[vi.name])

    model = shape_inference.infer_shapes(model)
    checker.check_model(model)
    onnx.save(model, FIXED)
    print(f"[OK] saved fixed ONNX: {FIXED}")


def run_onnxsim():
    try:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "onnxsim",
                str(FIXED),
                str(SIM),
            ],
            check=True,
        )
        print(f"[OK] saved simplified ONNX: {SIM}")
        return SIM
    except Exception as e:
        print(f"[WARN] onnxsim failed, will use fixed ONNX directly: {e}")
        return FIXED


def check_openvino(path: Path):
    core = ov.Core()
    model = core.read_model(str(path))

    model.reshape({
        "pixel_values": [1, 3, 256, 256],
    })

    print("\n[OpenVINO inputs]")
    for inp in model.inputs:
        print(inp.get_any_name(), inp.partial_shape)

    print("\n[OpenVINO outputs]")
    for out in model.outputs:
        print(out.get_any_name(), out.partial_shape)

    print("\n[Dynamic ops]")
    count = 0
    for op in model.get_ordered_ops():
        for out in op.outputs():
            if out.partial_shape.is_dynamic:
                print(op.get_type_name(), op.get_friendly_name(), out.partial_shape)
                count += 1
                break

    if count == 0:
        print("[OK] no dynamic ops detected")
    else:
        print(f"[WARN] dynamic op count: {count}")

    xml_path = path.with_suffix(".xml")
    ov.save_model(model, str(xml_path))
    print(f"\n[OK] saved OpenVINO IR: {xml_path}")

    print("\n[Compile CPU]")
    core.compile_model(model, "CPU")
    print("[OK] CPU compile success")

    print("\n[Compile GPU]")
    core.compile_model(model, "GPU")
    print("[OK] GPU compile success")


fix_onnx()
final_onnx = run_onnxsim()
check_openvino(final_onnx)